In [1]:
import torch
print(f"PyTorch: {torch.__version__}")

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset
from codecarbon import OfflineEmissionsTracker
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
import time, os, copy, json

os.makedirs("results", exist_ok=True)

SUBSET_SIZE = 5000
MAX_LEN = 128
BATCH_SIZE = 32
LR = 2e-5
DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

PyTorch: 2.12.1+cpu


C:\Users\bahar\source\repos\green-ai-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [3]:
#Veri Hazırlığı

dataset = load_dataset("nyu-mll/glue", "sst2")

# Subset
train_data = dataset["train"].shuffle(seed=42).select(range(SUBSET_SIZE))
val_data = dataset["validation"]  # 872 örnek, olduğu gibi

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length",
                     truncation=True, max_length=MAX_LEN)

train_data = train_data.map(tokenize, batched=True)
val_data = val_data.map(tokenize, batched=True)

train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

C:\Users\bahar\source\repos\green-ai-project\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bahar\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 107883.49 examples/s]
C:\Users\bahar\sou

Train: 5000, Val: 872


In [4]:
#Yardımcı Fonksiyonlar

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            inputs = {k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask"]}
            logits = model(**inputs).logits
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return acc, f1

def measure_inference(model, loader, n_batches=10):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            inputs = {k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask"]}
            t0 = time.perf_counter()
            _ = model(**inputs)
            times.append(time.perf_counter() - t0)
    return np.mean(times)

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        inputs = {k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask"]}
        outputs = model(**inputs, labels=batch["label"].to(DEVICE))
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)

results = []

In [5]:
# DENEY 1: Baseline Fine-Tuning
print("=" * 60)
print("DENEY 1: Baseline Fine-Tuning")
print("=" * 60)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

tracker = OfflineEmissionsTracker(
    project_name="distilbert_baseline",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

for epoch in range(3):
    loss = train_epoch(model, train_loader, optimizer)
    acc, f1 = evaluate(model, val_loader)
    print(f"  Epoch {epoch+1}/3 — Loss: {loss:.4f}, Acc: {acc:.4f}, F1: {f1:.4f}")

emissions = tracker.stop()

inf_time = measure_inference(model, val_loader)
final_acc, final_f1 = evaluate(model, val_loader)

results.append({
    "model": "DistilBERT",
    "experiment": "Baseline (3 epoch)",
    "accuracy": round(final_acc, 4),
    "f1_score": round(final_f1, 4),
    "model_size_mb": round(get_model_size_mb(model), 2),
    "parameters": count_params(model),
    "inference_time_s": round(inf_time, 4),
    "energy_kwh": emissions if emissions else 0,
    "epochs_trained": 3
})

# Baseline modeli sakla — diğer deneyler bundan türeyecek
baseline_state = copy.deepcopy(model.state_dict())
torch.save(baseline_state, "results/distilbert_baseline.pt")
print(f"\nBaseline — Acc: {final_acc:.4f}, Size: {get_model_size_mb(model):.2f} MB")
print(f"Energy: {emissions} kWh")

DENEY 1: Baseline Fine-Tuning


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5160.00it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[codecarbon INFO @ 14:06:12] offline tracker init
[codecarbon WARNING @ 14:06:12] Multiple instances of codecarbon are allowed to run at the same time

  Epoch 1/3 — Loss: 0.3892, Acc: 0.8704, F1: 0.8704
  Epoch 2/3 — Loss: 0.1907, Acc: 0.8739, F1: 0.8739
  Epoch 3/3 — Loss: 0.0955, Acc: 0.8589, F1: 0.8588

Baseline — Acc: 0.8589, Size: 255.42 MB
Energy: 0.009938628728455107 kWh


In [6]:
# DENEY 2: Dynamic Quantization
print("=" * 60)
print("DENEY 2: Dynamic Quantization (INT8)")
print("=" * 60)

# Baseline'dan yükle
model_q = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(DEVICE)
model_q.load_state_dict(baseline_state)

# Quantization — sadece inference optimize eder, training yok
tracker = OfflineEmissionsTracker(
    project_name="distilbert_quantized",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

model_q_int8 = torch.quantization.quantize_dynamic(
    model_q,
    {torch.nn.Linear},
    dtype=torch.qint8
)

emissions_q = tracker.stop()

# Quantized modelin boyutunu dosyadan ölç (in-memory size yanıltıcı olabilir)
torch.save(model_q_int8.state_dict(), "results/distilbert_quantized.pt")
quant_file_size = os.path.getsize("results/distilbert_quantized.pt") / (1024 ** 2)

acc_q, f1_q = evaluate(model_q_int8, val_loader)
inf_time_q = measure_inference(model_q_int8, val_loader)

results.append({
    "model": "DistilBERT",
    "experiment": "Dynamic Quantization INT8",
    "accuracy": round(acc_q, 4),
    "f1_score": round(f1_q, 4),
    "model_size_mb": round(quant_file_size, 2),
    "parameters": count_params(model_q),  # aynı kalır, ama dtype farklı
    "inference_time_s": round(inf_time_q, 4),
    "energy_kwh": emissions_q if emissions_q else 0,
    "epochs_trained": 0  # sadece inference optimizasyonu
})

print(f"Quantized — Acc: {acc_q:.4f}, File Size: {quant_file_size:.2f} MB")
print(f"Inference: {inf_time_q:.4f}s vs Baseline: {results[0]['inference_time_s']:.4f}s")

DENEY 2: Dynamic Quantization (INT8)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 842.07it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\bahar\AppData\Local\Temp\ipykernel_19204\2848565044.py:23: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10

Quantized — Acc: 0.8406, File Size: 132.29 MB
Inference: 2.8975s vs Baseline: 2.0713s


In [11]:
# DENEY 3: Structured Pruning (Attention Head Pruning)

import torch.nn as nn

def prune_attention_heads(attention, heads_to_remove):
    if len(heads_to_remove) == 0:
        return
    head_size = attention.attention_head_size
    n_heads = attention.n_heads
    heads_to_keep = sorted(set(range(n_heads)) - set(heads_to_remove))
    keep_indices = []
    for h in heads_to_keep:
        keep_indices.extend(range(h * head_size, (h + 1) * head_size))
    keep_indices = torch.tensor(keep_indices, dtype=torch.long)
    for proj in [attention.q_lin, attention.k_lin, attention.v_lin]:
        proj.weight = nn.Parameter(proj.weight.index_select(0, keep_indices).clone())
        proj.bias = nn.Parameter(proj.bias.index_select(0, keep_indices).clone())
        proj.out_features = len(keep_indices)
    attention.out_lin.weight = nn.Parameter(
        attention.out_lin.weight.index_select(1, keep_indices).clone()
    )
    attention.out_lin.in_features = len(keep_indices)
    attention.n_heads = len(heads_to_keep)
    attention.dim = len(heads_to_keep) * head_size

print("=" * 60)
print("DENEY 3: Structured Pruning — %33 Attention Head")
print("=" * 60)

model_p = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(DEVICE)
model_p.load_state_dict(baseline_state)

params_before = count_params(model_p)
size_before = get_model_size_mb(model_p)
print(f"  Pruning öncesi — Params: {params_before:,}, Size: {size_before:.2f} MB")

heads_to_prune = {i: [0, 2, 4, 6] for i in range(6)}

tracker = OfflineEmissionsTracker(
    project_name="distilbert_pruned",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

for layer_idx, head_indices in heads_to_prune.items():
    attn = model_p.distilbert.transformer.layer[layer_idx].attention
    prune_attention_heads(attn, set(head_indices))

optimizer_p = torch.optim.AdamW(model_p.parameters(), lr=1e-5)
loss_p = train_epoch(model_p, train_loader, optimizer_p)
print(f"  Recovery epoch — Loss: {loss_p:.4f}")

emissions_p = tracker.stop()

params_after = count_params(model_p)
size_after = get_model_size_mb(model_p)
acc_p, f1_p = evaluate(model_p, val_loader)
inf_time_p = measure_inference(model_p, val_loader)

results.append({
    "model": "DistilBERT",
    "experiment": "Structured Pruning 33% + 1ep recovery",
    "accuracy": round(acc_p, 4),
    "f1_score": round(f1_p, 4),
    "model_size_mb": round(size_after, 2),
    "parameters": params_after,
    "inference_time_s": round(inf_time_p, 4),
    "energy_kwh": emissions_p if emissions_p else 0,
    "epochs_trained": 1
})

print(f"  Pruning sonrası — Params: {params_after:,} ({100*(1 - params_after/params_before):.1f}% azalma)")
print(f"  Size: {size_after:.2f} MB (önceki: {size_before:.2f} MB)")
print(f"  Acc: {acc_p:.4f}, Inference: {inf_time_p:.4f}s")

DENEY 3: Structured Pruning — %33 Attention Head


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1145.56it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Pruning öncesi — Params: 66,955,010, Size: 255.42 MB
  Recovery epoch — Loss: 0.0861
  Pruning sonrası — Params: 62,231,810 (7.1% azalma)
  Size: 237.40 MB (önceki: 255.42 MB)
  Acc: 0.8624, Inference: 1.8575s


In [12]:
# DENEY 4: Early Stopping (max 8 epoch, patience 2)
print("=" * 60)
print("DENEY 4: Early Stopping (max 8 epoch, patience 2)")
print("=" * 60)

model_es = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(DEVICE)

optimizer_es = torch.optim.AdamW(model_es.parameters(), lr=LR)

tracker = OfflineEmissionsTracker(
    project_name="distilbert_early_stopping",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

best_val_loss = float("inf")
patience = 2
patience_counter = 0
best_state = None
stopped_epoch = 0

for epoch in range(8):
    loss = train_epoch(model_es, train_loader, optimizer_es)
    
    # Validation loss hesapla
    model_es.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask"]}
            out = model_es(**inputs, labels=batch["label"].to(DEVICE))
            val_losses.append(out.loss.item())
    val_loss = np.mean(val_losses)
    
    acc_es, f1_es = evaluate(model_es, val_loader)
    print(f"  Epoch {epoch+1}/8 — Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Acc: {acc_es:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = copy.deepcopy(model_es.state_dict())
        stopped_epoch = epoch + 1
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  >>> Early stopping at epoch {epoch+1}")
            break

emissions_es = tracker.stop()

# En iyi modeli yükle
model_es.load_state_dict(best_state)
final_acc_es, final_f1_es = evaluate(model_es, val_loader)
inf_time_es = measure_inference(model_es, val_loader)

results.append({
    "model": "DistilBERT",
    "experiment": f"Early Stopping (stopped ep {stopped_epoch})",
    "accuracy": round(final_acc_es, 4),
    "f1_score": round(final_f1_es, 4),
    "model_size_mb": round(get_model_size_mb(model_es), 2),
    "parameters": count_params(model_es),
    "inference_time_s": round(inf_time_es, 4),
    "energy_kwh": emissions_es if emissions_es else 0,
    "epochs_trained": stopped_epoch
})

print(f"\nBest epoch: {stopped_epoch}, Acc: {final_acc_es:.4f}")
print(f"Energy: {emissions_es} kWh")

DENEY 4: Early Stopping (max 8 epoch, patience 2)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4326.21it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/8 — Loss: 0.3865, Val Loss: 0.3166, Acc: 0.8555
  Epoch 2/8 — Loss: 0.1932, Val Loss: 0.3076, Acc: 0.8704
  Epoch 3/8 — Loss: 0.1072, Val Loss: 0.3468, Acc: 0.8647
  Epoch 4/8 — Loss: 0.0513, Val Loss: 0.4651, Acc: 0.8761
  >>> Early stopping at epoch 4

Best epoch: 2, Acc: 0.8704
Energy: 0.014899203119138489 kWh


In [13]:
# DistilBERT Sonuçları Kaydet
df_distilbert = pd.DataFrame(results)
df_distilbert.to_csv("results/distilbert_results.csv", index=False)
print(df_distilbert.to_string(index=False))

     model                            experiment  accuracy  f1_score  model_size_mb  parameters  inference_time_s  energy_kwh  epochs_trained
DistilBERT                    Baseline (3 epoch)    0.8589    0.8588         255.42    66955010            2.0713    0.009939               3
DistilBERT             Dynamic Quantization INT8    0.8406    0.8404         132.29    66955010            2.8975    0.000019               0
DistilBERT Structured Pruning 33% + 1ep recovery    0.8624    0.8621         237.40    62231810            1.8575    0.004980               1
DistilBERT         Early Stopping (stopped ep 2)    0.8704    0.8704         255.42    66955010            2.1066    0.014899               2


In [16]:
# Tüm Sonuçları Birleştir (MobileNet + DistilBERT)
df_mobilenet = pd.read_csv("../results/mobilenet_results.csv")
df_all = pd.concat([df_mobilenet, df_distilbert], ignore_index=True)
df_all.to_csv("../results/all_results.csv", index=False)

print("=" * 80)
print("TÜM SONUÇLAR")
print("=" * 80)
print(df_all.to_string(index=False))

TÜM SONUÇLAR
      Model                                 Teknik     F1  Precision  Recall  Accuracy  Eğitim Süresi (sn)  Enerji (kWh)  CO2 (kg)  Model Boyutu (MB)      model                            experiment  accuracy  f1_score  model_size_mb  parameters  inference_time_s  energy_kwh  epochs_trained
MobileNetV2              Baseline (Early Stopping) 0.6522     0.6550  0.6547    0.6547             1038.72      0.008409  0.003907               9.27        NaN                                   NaN       NaN       NaN            NaN         NaN               NaN         NaN             NaN
MobileNetV2                    Quantization (INT8) 0.6253     0.6336  0.6286    0.6286               19.93      0.000120  0.000056               2.60        NaN                                   NaN       NaN       NaN            NaN         NaN               NaN         NaN             NaN
MobileNetV2                 Pruning (%50 Sparsity) 0.6420     0.6434  0.6441    0.6441              377.24     

In [18]:
# Deney Metadata

metadata = {
    "distilbert_experiments": {
        "dataset": "SST-2 (GLUE)",
        "train_subset": SUBSET_SIZE,
        "val_size": len(val_data),
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "device": str(DEVICE),
        "pytorch_version": torch.__version__,
        "pruning_strategy": "Structured — attention head pruning, 4/12 heads per layer (33%)",
        "pruning_heads": {str(k): v for k, v in heads_to_prune.items()},
        "quantization": "Dynamic INT8 (torch.nn.Linear layers)",
        "early_stopping": {"max_epochs": 8, "patience": 2, "monitor": "val_loss"},
        "codecarbon": {"mode": "offline", "country": "TUR", "gpu_tracking": False}
    }
}

with open("../results/distilbert_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata kaydedildi: results/distilbert_metadata.json")

Metadata kaydedildi: results/distilbert_metadata.json
